# `new_uc` — one-call preset notebooks

This demo defines a small factory function, **`new_uc`**, that returns a ready-to-use
`UnichartNotebook` configured by a named **preset**. Instead of repeating the same
styling calls at the top of every analysis, you write:

```python
nb = new_uc("dark", data=df, set_name_column="ENGINE")
nb.plot(x="RPM", y="Temp")
```

Presets bundled in this demo:

| Preset | Look | Key settings |
|---|---|---|
| `default` | unichart out-of-the-box | matplotlib style, light mode |
| `dark` | dashboard on a dark background | dark mode, plotly style, vivid palette |
| `report` | static, document-friendly | static PNG output, muted palette, footer |
| `presentation` | big-screen slides | large fonts, thick lines, big markers |

Everything `new_uc` does uses the public `UnichartNotebook` API, so any preset value
can still be overridden afterwards on the returned notebook.

## Setup — imports and sample data

In [ ]:
# --- make repo-root importable (notebook lives in demo_notebooks/) ---
import sys, os
_repo_root = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

import numpy as np
import pandas as pd
from unichart import UnichartNotebook


def sample_data(seed=7):
    """Three 'engine runs': an RPM sweep with a few sensor channels each."""
    rng = np.random.default_rng(seed)
    rows = []
    for i, eng in enumerate(["Engine A", "Engine B", "Engine C"]):
        rpm = np.linspace(1000, 6000, 40)
        temp = 60 + 0.012 * rpm + 8 * i + rng.normal(0, 2.5, rpm.size)
        pressure = 14 + 0.004 * rpm + 1.5 * i + rng.normal(0, 0.8, rpm.size)
        thrust = 0.9 * (rpm / 1000) ** 1.8 + 2 * i + rng.normal(0, 0.6, rpm.size)
        rows.append(pd.DataFrame({
            "ENGINE": eng, "RPM": rpm, "Temp": temp,
            "Pressure": pressure, "Thrust": thrust,
        }))
    return pd.concat(rows, ignore_index=True)


df = sample_data()
df.head()

## The `new_uc` factory

`PRESETS` is a plain, user-editable dict: each entry lists the API calls / attributes
`new_uc` should apply to a fresh notebook. `new_uc(preset, data=...)` builds the
notebook, applies the preset, then (optionally) loads data — so datasets pick up the
preset's palette and markers at load time.

In [ ]:
PRESETS = {
    # unichart out-of-the-box: matplotlib look, light mode. Kept explicit so the
    # registry documents what "default" means.
    "default": {
        "plot_style": "matplotlib",
        "darkmode": False,
    },

    # Dark dashboard: plotly house style in dark mode with a vivid palette.
    "dark": {
        "plot_style": "plotly",
        "darkmode": True,
        "color_map": ["#00E5FF", "#FF4081", "#FFD740", "#69F0AE", "#B388FF"],
        "marker_map": ["o", "D", "s", "^", "v"],
        "font_sizes": {"suptitle": 22, "legend": 13, "axes_title": 15},
    },

    # Report/publication: flat PNGs keep the .ipynb (and exported HTML) small,
    # a muted palette prints well, and every figure carries a footer.
    "report": {
        "plot_style": "matplotlib",
        "darkmode": False,
        "static_images": {"enabled": True, "scale": 2},
        "copy_buttons": False,
        "color_map": ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B3"],
        "default_format": {"markersize": 5, "linewidth": 1.2},
        "figsize": (10, 6),
        "footer": "unichart report preset — data is synthetic",
    },

    # Presentation: readable from the back of the room.
    "presentation": {
        "plot_style": "matplotlib",
        "darkmode": False,
        "color_map": ["#E63946", "#457B9D", "#2A9D8F", "#F4A261"],
        "default_format": {"markersize": 11, "linewidth": 3.5, "edgewidth": 2},
        "font_sizes": {"all": 16, "suptitle": 28, "axes_title": 20, "legend": 18},
        "figsize": (14, 8),
    },
}


def new_uc(preset="default", data=None, **load_kwargs):
    """Return a new UnichartNotebook configured by a named preset.

    Parameters
    ----------
    preset : str
        A key of ``PRESETS`` (e.g. ``'default'``, ``'dark'``, ``'report'``,
        ``'presentation'``).
    data : DataFrame, optional
        If given, loaded via ``nb.load_df(data, **load_kwargs)`` *after* the
        preset is applied, so datasets inherit the preset's palette/markers.
    **load_kwargs
        Passed through to ``load_df`` (e.g. ``set_name_column='ENGINE'``).
    """
    try:
        cfg = PRESETS[preset]
    except KeyError:
        raise ValueError(f"Unknown preset {preset!r}. "
                         f"Available: {sorted(PRESETS)}") from None

    nb = UnichartNotebook()

    # Overall look first: plot style installs its own color map / format
    # defaults, so preset-specific palettes must be applied after it.
    if "plot_style" in cfg:
        nb.set_plot_style(cfg["plot_style"])
    nb.toggle_darkmode(cfg.get("darkmode", False))

    if "color_map" in cfg:
        nb.color_map = list(cfg["color_map"])
    if "marker_map" in cfg:
        nb.marker_map = list(cfg["marker_map"])
    if "default_format" in cfg:
        nb.set_default_format(**cfg["default_format"])
    if "figsize" in cfg:
        nb.set_default_format(figsize=cfg["figsize"])
    if "font_sizes" in cfg:
        nb.set_font_sizes(**cfg["font_sizes"])
    if "footer" in cfg:
        nb.footer = cfg["footer"]
    if "static_images" in cfg:
        nb.set_static_images(**cfg["static_images"])
    if "copy_buttons" in cfg:
        nb.set_copy_buttons(cfg["copy_buttons"])

    if data is not None:
        nb.load_df(data, **load_kwargs)
    return nb

## 1. `default` — the baseline

One call gives a notebook with data already loaded and split into per-engine datasets.

In [ ]:
nb = new_uc("default", data=df, set_name_column="ENGINE", set_idx_column="ENGINE")
nb.plot(x="RPM", y=["Temp", "Pressure"], suptitle="default preset")

## 2. `dark` — same data, different environment

Swapping the preset name is the only change: dark plotly template, vivid palette,
its own marker sequence and font sizes.

In [ ]:
nb_dark = new_uc("dark", data=df, set_name_column="ENGINE", set_idx_column="ENGINE")
nb_dark.plot(x="RPM", y=["Temp", "Pressure"], suptitle="dark preset")

## 3. `report` — static, document-friendly figures

This preset turns on static PNG output (requires `kaleido`; falls back to interactive
if it isn't installed), uses a muted print palette, and pins a footer on every figure.

In [ ]:
nb_report = new_uc("report", data=df, set_name_column="ENGINE", set_idx_column="ENGINE")
nb_report.plot(x="RPM", y="Thrust", suptitle="report preset")

## 4. `presentation` — readable from the back of the room

In [ ]:
nb_pres = new_uc("presentation", data=df, set_name_column="ENGINE", set_idx_column="ENGINE")
nb_pres.plot(x="RPM", y="Temp", suptitle="presentation preset")

## 5. Presets are starting points, not straitjackets

The returned object is an ordinary `UnichartNotebook`, so anything a preset set can be
tweaked afterwards — and you can register your own preset by adding a dict entry.

In [ ]:
# Post-tweak: take the dark notebook and recolor one dataset.
nb_dark.color(0, "white")
nb_dark.plot(x="RPM", y="Thrust", suptitle="dark preset, tweaked after the fact")

In [ ]:
# Your own preset: add an entry and use it immediately.
PRESETS["mono"] = {
    "plot_style": "matplotlib",
    "color_map": ["#222222", "#666666", "#AAAAAA"],
    "marker_map": ["o", "s", "^"],
    "default_format": {"markersize": 6, "linewidth": 2},
}

nb_mono = new_uc("mono", data=df, set_name_column="ENGINE", set_idx_column="ENGINE")
nb_mono.plot(x="RPM", y="Pressure", suptitle="custom 'mono' preset")